# Financial Data Classification with Explainable AI

**MSc Computer Science project — Kundeti Sai Chandra Sekhar**

This notebook implements the documented experimental workflow for retail financial data classification using the Walmart M5 dataset. It covers data preparation, feature engineering, TF-IDF NLP, four supervised classifiers, evaluation, and explainability using Random Forest feature importance and LIME.

> **Data note:** The original project used local/Google Drive dataset paths. Before running this notebook, place the required Walmart M5 CSV files in `../data/raw/`.


In [ ]:
# Environment setup and dataset import

from pathlib import Path
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

DATASET_PATH = Path("../data/raw")

sales = pd.read_csv(DATASET_PATH / "sales_train_validation.csv")
calendar = pd.read_csv(DATASET_PATH / "calendar.csv")
prices = pd.read_csv(DATASET_PATH / "sell_prices.csv")

print("Sales:", sales.shape)
print("Calendar:", calendar.shape)
print("Prices:", prices.shape)


In [ ]:

# SECTION 2 : EXPLORATORY DATA ANALYSIS (EDA)


print("="*100)
print("SECTION 2 : EXPLORATORY DATA ANALYSIS")
print("="*100)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# STEP 1 : DATASET OVERVIEW


print("\nDataset Shape")

print(f"Calendar Dataset : {calendar.shape}")
print(f"Sales Dataset    : {sales.shape}")
print(f"Price Dataset    : {prices.shape}")


# STEP 2 : DATASET INFORMATION


print("\nCalendar Dataset Information")

display(calendar.head())

print("\nSales Dataset Information")

display(sales.head())

print("\nPrice Dataset Information")

display(prices.head())


# STEP 3 : MISSING VALUES


print("\nMissing Values")

missing = pd.DataFrame({

    "Dataset":["Calendar","Sales","Prices"],

    "Missing Values":[

        calendar.isnull().sum().sum(),

        sales.isnull().sum().sum(),

        prices.isnull().sum().sum()

    ]

})

display(missing)


# GRAPH 1 : MISSING VALUES


plt.figure(figsize=(8,5))

bars = plt.bar(

    missing["Dataset"],

    missing["Missing Values"],

    color=["#3498DB","#E74C3C","#2ECC71"]

)

plt.title("Missing Values Across Datasets")

plt.ylabel("Number of Missing Values")

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        f"{int(bar.get_height()):,}",

        ha="center"

    )

plt.show()


# STEP 4 : DESCRIPTIVE STATISTICS


print("\nSelling Price Statistics")

display(prices[["sell_price"]].describe())


# GRAPH 2 : SELLING PRICE DISTRIBUTION


plt.figure(figsize=(8,5))

plt.hist(

    prices["sell_price"],

    bins=30,

    color="#9B59B6",

    edgecolor="black"

)

plt.title("Distribution of Selling Price")

plt.xlabel("Selling Price")

plt.ylabel("Frequency")

plt.grid(alpha=0.3)

plt.show()


# STEP 5 : PRODUCT CATEGORY DISTRIBUTION


category_count = sales["cat_id"].value_counts()

print("\nCategory Distribution")

display(category_count)


# GRAPH 3 : PRODUCT CATEGORY DISTRIBUTION


plt.figure(figsize=(7,5))

bars = plt.bar(

    category_count.index,

    category_count.values,

    color=["#16A085","#F39C12","#2980B9"]

)

plt.title("Product Category Distribution")

plt.ylabel("Number of Products")

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        str(int(bar.get_height())),

        ha="center"

    )

plt.show()


# STEP 6 : CORRELATION ANALYSIS (NEW)


print("\nGenerating Correlation Heatmap...")

numeric_columns = prices.select_dtypes(include=["number"])

correlation = numeric_columns.corr()

display(correlation.round(3))


# GRAPH 4 : CORRELATION HEATMAP


plt.figure(figsize=(8,6))

plt.imshow(

    correlation,

    cmap="coolwarm",

    aspect="auto"

)

plt.colorbar()

plt.xticks(

    range(len(correlation.columns)),

    correlation.columns,

    rotation=45,

    ha="right"

)

plt.yticks(

    range(len(correlation.columns)),

    correlation.columns

)

plt.title("Correlation Heatmap of Numerical Features")

for i in range(len(correlation.columns)):
    for j in range(len(correlation.columns)):
        plt.text(

            j,

            i,

            f"{correlation.iloc[i,j]:.2f}",

            ha="center",

            va="center",

            fontsize=8,

            color="black"

        )

plt.tight_layout()

plt.show()


# STEP 7 : DATA PREVIEW


print("\nCalendar Dataset Preview")

display(calendar.head())

print("\nSales Dataset Preview")

display(sales.head())

print("\nPrice Dataset Preview")

display(prices.head())

print("="*100)
print("SECTION 2 COMPLETED")
print("="*100)

SECTION 2 : EXPLORATORY DATA ANALYSIS

Dataset Shape


NameError: name 'calendar' is not defined

In [ ]:


# SECTION 3 : DATA CLEANING AND DATA INTEGRATION


print("="*100)
print("SECTION 3 : DATA CLEANING AND DATA INTEGRATION")
print("="*100)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# STEP 1 : SELECT LAST 365 DAYS OF SALES DATA


print("\nSelecting the Most Recent 365 Days...")

id_columns = [

    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"

]

sales_columns = sales.columns[-365:]

sales_subset = sales[id_columns + list(sales_columns)]

print("Selected Dataset Shape")

print(sales_subset.shape)


# STEP 2 : CREATE A REPRESENTATIVE SAMPLE


print("\nCreating Representative Sample...")

sales_subset = sales_subset.sample(

    n=1000,

    random_state=42

)

print("Sample Shape")

print(sales_subset.shape)


# STEP 3 : CONVERT TO LONG FORMAT


print("\nConverting to Long Format...")

merged_sales = pd.melt(

    sales_subset,

    id_vars=id_columns,

    var_name="d",

    value_name="sales"

)

print("Long Format Shape")

print(merged_sales.shape)


# STEP 4 : MERGE CALENDAR


print("\nMerging Calendar Dataset...")

calendar_subset = calendar[

    calendar["d"].isin(

        merged_sales["d"].unique()

    )

]

merged_data = merged_sales.merge(

    calendar_subset,

    on="d",

    how="left"

)

print("Calendar Merge Completed")

print(merged_data.shape)


# STEP 5 : MERGE SELL PRICES


print("\nMerging Selling Price Dataset...")

merged_data = merged_data.merge(

    prices,

    on=[

        "store_id",

        "item_id",

        "wm_yr_wk"

    ],

    how="left"

)

print("Price Merge Completed")

print(merged_data.shape)


# STEP 6 : HANDLE MISSING VALUES


print("\nHandling Missing Values...")

missing_before = merged_data.isnull().sum().sum()

merged_data["sell_price"] = merged_data["sell_price"].fillna(

    merged_data["sell_price"].median()

)

merged_data.fillna("Unknown", inplace=True)

missing_after = merged_data.isnull().sum().sum()

print("Missing Before :", missing_before)

print("Missing After :", missing_after)


# STEP 7 : REMOVE DUPLICATES


duplicates_before = merged_data.duplicated().sum()

merged_data = merged_data.drop_duplicates()

duplicates_after = merged_data.duplicated().sum()

print("\nDuplicates Before :", duplicates_before)

print("Duplicates After :", duplicates_after)


# STEP 8 : DATASET SUMMARY


summary = pd.DataFrame({

    "Description":[

        "Rows",

        "Columns",

        "Missing Values",

        "Duplicate Records"

    ],

    "Value":[

        merged_data.shape[0],

        merged_data.shape[1],

        merged_data.isnull().sum().sum(),

        merged_data.duplicated().sum()

    ]

})

print("\nMerged Dataset Summary")

display(summary)


# GRAPH 1 : DATASET TRANSFORMATION


transformation = pd.DataFrame({

    "Stage":[

        "Original Sales",

        "Sample",

        "Merged Dataset"

    ],

    "Rows":[

        sales.shape[0],

        sales_subset.shape[0],

        merged_data.shape[0]

    ]

})

plt.figure(figsize=(10,5))

bars = plt.bar(

    transformation["Stage"],

    transformation["Rows"],

    color=["#2E86AB","#F6AA1C","#6A4C93"]

)

plt.title("Dataset Transformation Process")

plt.ylabel("Number of Records")

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        f"{int(bar.get_height()):,}",

        ha="center"

    )

plt.show()


# GRAPH 2 : MISSING VALUES


missing = pd.DataFrame({

    "Stage":[

        "Before Cleaning",

        "After Cleaning"

    ],

    "Missing Values":[

        missing_before,

        missing_after

    ]

})

plt.figure(figsize=(8,5))

bars = plt.bar(

    missing["Stage"],

    missing["Missing Values"],

    color=["#D62828","#2A9D8F"]

)

plt.title("Missing Values Before and After Cleaning")

plt.ylabel("Number of Missing Values")

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        f"{int(bar.get_height()):,}",

        ha="center"

    )

plt.show()


# STEP 9 : PREVIEW FINAL DATASET


print("\nMerged Dataset Preview")

display(merged_data.head())

print("="*100)
print("SECTION 3 COMPLETED")
print("="*100)

In [ ]:

# SECTION 4 : FEATURE ENGINEERING


print("="*100)
print("SECTION 4 : FEATURE ENGINEERING")
print("="*100)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder


# STEP 1 : VERIFY DATASET


print("\nMerged Dataset Shape")

print(merged_data.shape)


# STEP 2 : CREATE DATE FEATURES


print("\nCreating Date Features...")

merged_data["date"] = pd.to_datetime(merged_data["date"])

merged_data["Year"] = merged_data["date"].dt.year

merged_data["Month"] = merged_data["date"].dt.month

merged_data["Week"] = merged_data["date"].dt.isocalendar().week.astype(int)

merged_data["Day"] = merged_data["date"].dt.day

merged_data["Day_Name"] = merged_data["date"].dt.day_name()

merged_data["Weekend"] = np.where(

    merged_data["Day_Name"].isin(

        ["Saturday","Sunday"]

    ),

    1,

    0

)

print("Date Features Created Successfully")


# STEP 3 : CREATE PRICE CATEGORY


merged_data["Price_Category"] = pd.cut(

    merged_data["sell_price"],

    bins=[

        0,

        5,

        15,

        30,

        np.inf

    ],

    labels=[

        "Low",

        "Medium",

        "High",

        "Premium"

    ]

)

print("Price Category Created Successfully")


# STEP 4 : SELECT FEATURES


dataset = merged_data[[

    "sell_price",

    "Month",

    "Week",

    "Day",

    "Weekend",

    "snap_CA",

    "snap_TX",

    "snap_WI",

    "state_id",

    "store_id",

    "dept_id",

    "Price_Category"

]].copy()

print("\nDataset Shape")

print(dataset.shape)


# STEP 5 : HANDLE MISSING VALUES


dataset["sell_price"] = dataset["sell_price"].fillna(

    dataset["sell_price"].median()

)

dataset["Price_Category"] = dataset["Price_Category"].astype(str)

print("Missing Values Removed")


# STEP 6 : LABEL ENCODING


encoders = {}

categorical_columns = [

    "state_id",

    "store_id",

    "dept_id",

    "Price_Category"

]

summary = []

for column in categorical_columns:

    encoder = LabelEncoder()

    dataset[column] = encoder.fit_transform(

        dataset[column]

    )

    encoders[column] = encoder

    summary.append({

        "Feature":column,

        "Unique Classes":len(encoder.classes_)

    })

encoding_summary = pd.DataFrame(summary)

print("\nEncoding Summary")

display(encoding_summary)


# STEP 7 : FEATURE MATRIX


X = dataset.drop("dept_id",axis=1)

y = dataset["dept_id"]

print("\nFeature Matrix")

print(X.shape)

print("\nTarget Variable")

print(y.shape)


# GRAPH 1 : DEPARTMENT DISTRIBUTION


department = merged_data["dept_id"].value_counts()

plt.figure(figsize=(10,6))

bars = plt.bar(

    department.index,

    department.values,

    color=[

        "#003f5c",

        "#58508d",

        "#bc5090",

        "#ff6361",

        "#ffa600",

        "#2ca02c",

        "#17becf"

    ]

)

plt.title("Distribution of Walmart Departments")

plt.xlabel("Department")

plt.ylabel("Frequency")

plt.xticks(rotation=45)

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        format(int(bar.get_height()),","),

        ha="center",

        fontsize=8

    )

plt.show()

display(department)


# GRAPH 2 : PRICE CATEGORY


price = merged_data["Price_Category"].value_counts()

plt.figure(figsize=(8,5))

bars = plt.bar(

    price.index,

    price.values,

    color=[

        "#2E86AB",

        "#F6AA1C",

        "#A23B72",

        "#1B998B"

    ]

)

plt.title("Distribution of Product Price Categories")

plt.xlabel("Price Category")

plt.ylabel("Frequency")

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        format(int(bar.get_height()),","),

        ha="center"

    )

plt.show()

display(price)


# FEATURE SUMMARY


summary = pd.DataFrame({

    "Feature":dataset.columns,

    "Data Type":dataset.dtypes.values,

    "Missing Values":dataset.isnull().sum().values,

    "Unique Values":dataset.nunique().values

})

print("\nFeature Summary")

display(summary)

print("="*100)
print("SECTION 4 COMPLETED")
print("="*100)

In [ ]:

# SECTION 5 : NATURAL LANGUAGE PROCESSING (NLP) USING TF-IDF


print("="*100)
print("SECTION 5 : NATURAL LANGUAGE PROCESSING (NLP)")
print("="*100)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
from collections import Counter


# STEP 1 : VERIFY DATASET


if "dataset" not in globals():
    raise Exception("Please run Section 4 before Section 5.")

print("\nCurrent Dataset Shape")
print(dataset.shape)


# STEP 2 : CREATE COMBINED TEXT COLUMN


print("\nCreating Combined Text Feature...")

merged_data["Text"] = (

    merged_data["event_name_1"].fillna("No_Event").astype(str)

    + " " +

    merged_data["event_name_2"].fillna("No_Event").astype(str)

    + " " +

    merged_data["event_type_1"].fillna("None").astype(str)

    + " " +

    merged_data["event_type_2"].fillna("None").astype(str)

)

print("Combined Text Created Successfully")


# STEP 3 : TEXT CLEANING


print("\nCleaning Text...")

merged_data["Text"] = (

    merged_data["Text"]

    .str.lower()

    .str.replace(r"[^a-z ]"," ",regex=True)

    .str.replace(r"\s+"," ",regex=True)

    .str.strip()

)

print("Text Cleaning Completed")


# STEP 4 : TF-IDF FEATURE EXTRACTION


print("\nExtracting TF-IDF Features...")

tfidf = TfidfVectorizer(

    stop_words="english",

    max_features=15

)

tfidf_matrix = tfidf.fit_transform(

    merged_data["Text"]

)

tfidf_df = pd.DataFrame(

    tfidf_matrix.toarray(),

    columns=tfidf.get_feature_names_out()

)

print("TF-IDF Matrix Shape")

print(tfidf_df.shape)


# STEP 5 : MERGE TF-IDF FEATURES


print("\nIntegrating NLP Features...")

dataset = pd.concat(

    [

        dataset.reset_index(drop=True),

        tfidf_df.reset_index(drop=True)

    ],

    axis=1

)

print("Updated Dataset Shape")

print(dataset.shape)


# GRAPH 1 : TF-IDF FEATURE IMPORTANCE


importance = tfidf_df.mean().sort_values(

    ascending=True

)

plt.figure(figsize=(11,6))

bars = plt.barh(

    importance.index,

    importance.values,

    color="#008B8B"

)

plt.title(

    "Average TF-IDF Score of Extracted Terms",

    fontsize=14,

    fontweight="bold"

)

plt.xlabel("Average TF-IDF Score")

plt.grid(axis="x",alpha=0.3)

for i,v in enumerate(importance.values):

    plt.text(

        v,

        i,

        f"{v:.3f}",

        va="center",

        fontsize=8

    )

plt.show()

tfidf_summary = importance.reset_index()

tfidf_summary.columns=[

    "Term",

    "Average TF-IDF Score"

]

print("\nAverage TF-IDF Scores")

display(tfidf_summary)


# GRAPH 2 : WORD CLOUD


text = " ".join(

    merged_data["Text"]

)

wordcloud = WordCloud(

    width=1200,

    height=600,

    background_color="white",

    colormap="viridis"

).generate(text)

plt.figure(figsize=(13,6))

plt.imshow(wordcloud)

plt.axis("off")

plt.title(

    "Word Cloud of Walmart Event Information",

    fontsize=14,

    fontweight="bold"

)

plt.show()


# TOP WORD FREQUENCY


words = text.split()

frequency = Counter(words)

top_words = pd.DataFrame(

    frequency.most_common(15),

    columns=[

        "Word",

        "Frequency"

    ]

)

print("\nTop 15 Most Frequent Words")

display(top_words)




# TF-IDF FEATURE SUMMARY


summary = pd.DataFrame({

    "Description":[

        "Documents",

        "Vocabulary Size",

        "TF-IDF Features",

        "Final Dataset Columns"

    ],

    "Value":[

        len(merged_data),

        len(tfidf.vocabulary_),

        tfidf_df.shape[1],

        dataset.shape[1]

    ]

})

print("\nNLP Summary")

display(summary)

print("="*100)
print("SECTION 5 COMPLETED")
print("="*100)

In [ ]:

# SECTION 6 : DATA PREPROCESSING


print("="*100)
print("SECTION 6 : DATA PREPROCESSING")
print("="*100)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# STEP 1 : VERIFY DATASET


if "dataset" not in globals():
    raise Exception("Please run Section 5 before Section 6.")

print("\nDataset Shape")

print(dataset.shape)


# STEP 2 : DEFINE FEATURES AND TARGET


print("\nPreparing Feature Matrix...")

X = dataset.drop("dept_id",axis=1)

y = dataset["dept_id"]

print("Feature Matrix :",X.shape)

print("Target Variable :",y.shape)


# STEP 3 : IDENTIFY NUMERICAL FEATURES


numerical_features = [

    "sell_price",

    "Month",

    "Week",

    "Day"

]

print("\nNumerical Features")

print(numerical_features)


# STEP 4 : STANDARDIZATION


print("\nApplying Standard Scaling...")

scaler = StandardScaler()

X[numerical_features] = scaler.fit_transform(

    X[numerical_features]

)

print("Scaling Completed Successfully")


# STEP 5 : TRAIN TEST SPLIT


print("\nSplitting Dataset...")

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print("Training Dataset :",X_train.shape)

print("Testing Dataset :",X_test.shape)


# STEP 6 : TRAIN TEST SUMMARY


summary = pd.DataFrame({

    "Dataset":[

        "Training",

        "Testing"

    ],

    "Records":[

        len(X_train),

        len(X_test)

    ],

    "Percentage":[

        round(len(X_train)/len(dataset)*100,2),

        round(len(X_test)/len(dataset)*100,2)

    ]

})

print("\nTrain-Test Summary")

display(summary)


# GRAPH 1 : TRAIN TEST SPLIT


plt.figure(figsize=(8,5))

bars = plt.bar(

    summary["Dataset"],

    summary["Records"],

    color=[

        "#E67E22",

        "#1ABC9C"

    ]

)

plt.title(

    "Training and Testing Dataset Distribution",

    fontsize=14,

    fontweight="bold"

)

plt.ylabel("Number of Records")

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        format(int(bar.get_height()),","),

        ha="center",

        fontsize=10

    )

plt.show()


# GRAPH 2 : TARGET CLASS DISTRIBUTION


target = y.value_counts().sort_index()

colors = [

    "#003F5C",

    "#58508D",

    "#BC5090",

    "#FF6361",

    "#FFA600",

    "#2CA02C",

    "#17BECF"

]

plt.figure(figsize=(10,6))

bars = plt.bar(

    target.index.astype(str),

    target.values,

    color=colors[:len(target)]

)

plt.title(

    "Distribution of Target Classes",

    fontsize=14,

    fontweight="bold"

)

plt.xlabel("Department")

plt.ylabel("Frequency")

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        format(int(bar.get_height()),","),

        ha="center",

        fontsize=8

    )

plt.show()

target_summary = target.reset_index()

target_summary.columns=[

    "Department",

    "Frequency"

]

print("\nTarget Distribution")

display(target_summary)

# STEP 7 : DATASET VERIFICATION


verification = pd.DataFrame({

    "Description":[

        "Total Features",

        "Training Records",

        "Testing Records",

        "Target Classes"

    ],

    "Value":[

        X.shape[1],

        X_train.shape[0],

        X_test.shape[0],

        y.nunique()

    ]

})

print("\nDataset Verification")

display(verification)

print("="*100)
print("SECTION 6 COMPLETED")
print("="*100)

In [ ]:

# SECTION 7 : LOGISTIC REGRESSION CLASSIFICATION


print("="*100)
print("SECTION 7 : LOGISTIC REGRESSION CLASSIFICATION")
print("="*100)

import time
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# STEP 1 : TRAIN MODEL


print("\nTraining Logistic Regression Model...")

start_train = time.time()

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    solver="lbfgs"
)

logistic_model.fit(X_train, y_train)

training_time_lr = time.time() - start_train

print("Training Completed")


# STEP 2 : MODEL PREDICTION


print("\nGenerating Predictions...")

start_predict = time.time()

y_pred_lr = logistic_model.predict(X_test)

prediction_time_lr = time.time() - start_predict

print("Prediction Completed")


# STEP 3 : MODEL PERFORMANCE


lr_accuracy = accuracy_score(y_test, y_pred_lr)

lr_precision = precision_score(
    y_test,
    y_pred_lr,
    average="weighted"
)

lr_recall = recall_score(
    y_test,
    y_pred_lr,
    average="weighted"
)

lr_f1 = f1_score(
    y_test,
    y_pred_lr,
    average="weighted"
)

logistic_results = pd.DataFrame({

    "Evaluation Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Score":[
        round(lr_accuracy,4),
        round(lr_precision,4),
        round(lr_recall,4),
        round(lr_f1,4)
    ]

})

print("\nLogistic Regression Performance")

display(logistic_results)


# STEP 4 : CLASSIFICATION REPORT


print("\nClassification Report")

print(classification_report(y_test, y_pred_lr))


# GRAPH 1 : CONFUSION MATRIX


fig, ax = plt.subplots(figsize=(8,6))

ConfusionMatrixDisplay.from_predictions(

    y_test,

    y_pred_lr,

    cmap="Blues",

    colorbar=False,

    ax=ax

)

plt.title("Confusion Matrix - Logistic Regression")

plt.grid(False)

plt.show()


# GRAPH 2 : PERFORMANCE METRICS


metric_colors = [

    "#2E86AB",
    "#F6AA1C",
    "#A23B72",
    "#1B998B"

]

plt.figure(figsize=(8,5))

bars = plt.bar(

    logistic_results["Evaluation Metric"],

    logistic_results["Score"],

    color=metric_colors

)

plt.ylim(0,1.05)

plt.title("Logistic Regression Performance Metrics")

plt.ylabel("Score")

plt.grid(axis="y", alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        f"{bar.get_height():.3f}",

        ha="center",

        fontsize=10

    )

plt.show()


# STEP 5 : SAMPLE PREDICTIONS


prediction_table = pd.DataFrame({

    "Actual Department": y_test.values[:15],

    "Predicted Department": y_pred_lr[:15]

})

print("\nSample Predictions")

display(prediction_table)


# STEP 6 : MODEL EXECUTION SUMMARY


execution_summary = pd.DataFrame({

    "Description":[

        "Training Time (Seconds)",

        "Prediction Time (Seconds)",

        "Training Records",

        "Testing Records"

    ],

    "Value":[

        round(training_time_lr,4),

        round(prediction_time_lr,4),

        X_train.shape[0],

        X_test.shape[0]

    ]

})

print("\nExecution Summary")

display(execution_summary)


# STORE RESULTS FOR MODEL COMPARISON


model_results = []

model_results.append({

    "Model":"Logistic Regression",

    "Accuracy":lr_accuracy,

    "Precision":lr_precision,

    "Recall":lr_recall,

    "F1 Score":lr_f1,

    "Training Time":training_time_lr,

    "Prediction Time":prediction_time_lr

})

print("="*100)
print("SECTION 7 COMPLETED")
print("="*100)

In [ ]:

# SECTION 8 : DECISION TREE CLASSIFICATION


print("="*100)
print("SECTION 8 : DECISION TREE CLASSIFICATION")
print("="*100)

import time
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)


# STEP 1 : TRAIN DECISION TREE MODEL


print("\nTraining Decision Tree Model...")

start_train = time.time()

decision_tree_model = DecisionTreeClassifier(

    criterion="gini",
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42

)

decision_tree_model.fit(X_train, y_train)

training_time_dt = time.time() - start_train

print("Decision Tree Training Completed")


# STEP 2 : MODEL PREDICTION


print("\nGenerating Predictions...")

start_predict = time.time()

y_pred_dt = decision_tree_model.predict(X_test)

prediction_time_dt = time.time() - start_predict

print("Prediction Completed")


# STEP 3 : PERFORMANCE EVALUATION


dt_accuracy = accuracy_score(y_test, y_pred_dt)

dt_precision = precision_score(
    y_test,
    y_pred_dt,
    average="weighted"
)

dt_recall = recall_score(
    y_test,
    y_pred_dt,
    average="weighted"
)

dt_f1 = f1_score(
    y_test,
    y_pred_dt,
    average="weighted"
)

decision_tree_results = pd.DataFrame({

    "Evaluation Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Score":[
        round(dt_accuracy,4),
        round(dt_precision,4),
        round(dt_recall,4),
        round(dt_f1,4)
    ]

})

print("\nDecision Tree Performance")

display(decision_tree_results)


# STEP 4 : CLASSIFICATION REPORT


print("\nClassification Report")

print(classification_report(y_test, y_pred_dt))


# GRAPH 1 : CONFUSION MATRIX


fig, ax = plt.subplots(figsize=(8,6))

ConfusionMatrixDisplay.from_predictions(

    y_test,

    y_pred_dt,

    cmap="Oranges",

    colorbar=False,

    ax=ax

)

plt.title("Confusion Matrix - Decision Tree")

plt.grid(False)

plt.show()


# STEP 5 : FEATURE IMPORTANCE


feature_importance = pd.DataFrame({

    "Feature": X.columns,
    "Importance": decision_tree_model.feature_importances_

})

feature_importance = feature_importance.sort_values(

    by="Importance",
    ascending=False

)

top_features = feature_importance.head(10)


# GRAPH 2 : TOP FEATURE IMPORTANCE


plt.figure(figsize=(10,6))

bars = plt.barh(

    top_features["Feature"],

    top_features["Importance"],

    color="#8E44AD"

)

plt.gca().invert_yaxis()

plt.title("Top 10 Important Features - Decision Tree")

plt.xlabel("Importance Score")

plt.grid(axis="x", alpha=0.3)

for i, value in enumerate(top_features["Importance"]):

    plt.text(

        value,

        i,

        f"{value:.3f}",

        va="center",

        fontsize=9

    )

plt.show()

print("\nTop Feature Importance")

display(top_features)


# STEP 6 : SAMPLE PREDICTIONS


prediction_table = pd.DataFrame({

    "Actual Department": y_test.values[:15],
    "Predicted Department": y_pred_dt[:15]

})

print("\nSample Predictions")

display(prediction_table)


# STEP 7 : EXECUTION SUMMARY


execution_summary = pd.DataFrame({

    "Description":[

        "Training Time (Seconds)",

        "Prediction Time (Seconds)",

        "Tree Depth",

        "Number of Leaves"

    ],

    "Value":[

        round(training_time_dt,4),

        round(prediction_time_dt,4),

        decision_tree_model.get_depth(),

        decision_tree_model.get_n_leaves()

    ]

})

print("\nExecution Summary")

display(execution_summary)


# STEP 8 : SAVE RESULTS FOR MODEL COMPARISON


model_results.append({

    "Model":"Decision Tree",

    "Accuracy":dt_accuracy,

    "Precision":dt_precision,

    "Recall":dt_recall,

    "F1 Score":dt_f1,

    "Training Time":training_time_dt,

    "Prediction Time":prediction_time_dt

})

print("="*100)
print("SECTION 8 COMPLETED")
print("="*100)

In [ ]:

# SECTION 9 : RANDOM FOREST CLASSIFICATION (OPTIMISED VERSION)


print("="*100)
print("SECTION 9 : RANDOM FOREST CLASSIFICATION")
print("="*100)

import time
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)


# STEP 1 : BUILD RANDOM FOREST MODEL


print("\nBuilding Random Forest Model...")

rf_model = RandomForestClassifier(

    n_estimators=100,

    max_depth=20,

    min_samples_split=5,

    min_samples_leaf=2,

    max_features="sqrt",

    bootstrap=True,

    random_state=42,

    n_jobs=-1

)


# STEP 2 : TRAIN MODEL


print("\nTraining Random Forest...")

start_train = time.time()

rf_model.fit(X_train, y_train)

training_time_rf = time.time() - start_train

print("Training Completed")


# STEP 3 : PREDICTION


print("\nPredicting Test Dataset...")

start_prediction = time.time()

y_pred_rf = rf_model.predict(X_test)

prediction_time_rf = time.time() - start_prediction

print("Prediction Completed")


# STEP 4 : PERFORMANCE


rf_accuracy = accuracy_score(y_test, y_pred_rf)

rf_precision = precision_score(

    y_test,

    y_pred_rf,

    average="weighted"

)

rf_recall = recall_score(

    y_test,

    y_pred_rf,

    average="weighted"

)

rf_f1 = f1_score(

    y_test,

    y_pred_rf,

    average="weighted"

)

training_accuracy = rf_model.score(X_train, y_train)

testing_accuracy = rf_model.score(X_test, y_test)


# STEP 5 : LIGHTWEIGHT CROSS VALIDATION


print("\nRunning Cross Validation...")

sample_size = min(30000, len(X_train))

X_cv = X_train.sample(

    n=sample_size,

    random_state=42

)

y_cv = y_train.loc[X_cv.index]

cv_scores = cross_val_score(

    rf_model,

    X_cv,

    y_cv,

    cv=3,

    scoring="accuracy",

    n_jobs=-1

)

cross_validation = cv_scores.mean()


# STEP 6 : PERFORMANCE TABLE


rf_results = pd.DataFrame({

    "Metric":[

        "Training Accuracy",

        "Testing Accuracy",

        "Cross Validation",

        "Precision",

        "Recall",

        "F1 Score"

    ],

    "Score":[

        training_accuracy,

        testing_accuracy,

        cross_validation,

        rf_precision,

        rf_recall,

        rf_f1

    ]

})

rf_results["Score"] = rf_results["Score"].round(4)

print("\nRandom Forest Performance")

display(rf_results)


# STEP 7 : CLASSIFICATION REPORT


print("\nClassification Report")

print(classification_report(y_test, y_pred_rf))


# GRAPH 1 : CONFUSION MATRIX


fig, ax = plt.subplots(figsize=(8,6))

ConfusionMatrixDisplay.from_predictions(

    y_test,

    y_pred_rf,

    cmap="Greens",

    colorbar=False,

    ax=ax

)

plt.title("Confusion Matrix - Random Forest")

plt.grid(False)

plt.show()


# STEP 8 : FEATURE IMPORTANCE


importance = pd.DataFrame({

    "Feature": X.columns,

    "Importance": rf_model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

top10 = importance.head(10)


# GRAPH 2 : FEATURE IMPORTANCE


plt.figure(figsize=(10,6))

bars = plt.barh(

    top10["Feature"],

    top10["Importance"],

    color="#C0392B"

)

plt.gca().invert_yaxis()

plt.title("Top 10 Feature Importance - Random Forest")

plt.xlabel("Importance")

plt.grid(axis="x", alpha=0.3)

for i, value in enumerate(top10["Importance"]):

    plt.text(

        value,

        i,

        f"{value:.3f}",

        va="center"

    )

plt.show()

print("\nTop 10 Important Features")

display(top10)


# GRAPH 3 : TRAINING VS TESTING ACCURACY


accuracy_df = pd.DataFrame({

    "Dataset":[

        "Training",

        "Testing"

    ],

    "Accuracy":[

        training_accuracy,

        testing_accuracy

    ]

})

plt.figure(figsize=(7,5))

bars = plt.bar(

    accuracy_df["Dataset"],

    accuracy_df["Accuracy"],

    color=["#27AE60","#F39C12"]

)

plt.title("Training vs Testing Accuracy")

plt.ylabel("Accuracy")

plt.ylim(0,1.05)

plt.grid(axis="y", alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        f"{bar.get_height():.3f}",

        ha="center"

    )

plt.show()


# STEP 9 : EXECUTION SUMMARY


execution = pd.DataFrame({

    "Description":[

        "Training Time (Seconds)",

        "Prediction Time (Seconds)",

        "Cross Validation Accuracy",

        "Number of Trees",

        "Maximum Depth"

    ],

    "Value":[

        round(training_time_rf,3),

        round(prediction_time_rf,4),

        round(cross_validation,4),

        rf_model.n_estimators,

        rf_model.max_depth

    ]

})

print("\nExecution Summary")

display(execution)


# STEP 10 : SAVE RESULTS


model_results.append({

    "Model":"Random Forest",

    "Accuracy":rf_accuracy,

    "Precision":rf_precision,

    "Recall":rf_recall,

    "F1 Score":rf_f1,

    "Training Time":training_time_rf,

    "Prediction Time":prediction_time_rf,

    "Cross Validation":cross_validation

})

print("="*100)
print("SECTION 9 COMPLETED")
print("="*100)










In [ ]:

# SECTION 10 : GAUSSIAN NAÏVE BAYES CLASSIFICATION


print("="*100)
print("SECTION 10 : GAUSSIAN NAÏVE BAYES CLASSIFICATION")
print("="*100)

import time
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)


# STEP 1 : TRAIN GAUSSIAN NAÏVE BAYES MODEL


print("\nTraining Gaussian Naïve Bayes Model...")

start_train = time.time()

nb_model = GaussianNB()

nb_model.fit(X_train, y_train)

training_time_nb = time.time() - start_train

print("Training Completed Successfully")


# STEP 2 : MODEL PREDICTION


print("\nGenerating Predictions...")

start_prediction = time.time()

y_pred_nb = nb_model.predict(X_test)

prediction_time_nb = time.time() - start_prediction

print("Prediction Completed")


# STEP 3 : PERFORMANCE EVALUATION


nb_accuracy = accuracy_score(y_test, y_pred_nb)

nb_precision = precision_score(
    y_test,
    y_pred_nb,
    average="weighted"
)

nb_recall = recall_score(
    y_test,
    y_pred_nb,
    average="weighted"
)

nb_f1 = f1_score(
    y_test,
    y_pred_nb,
    average="weighted"
)

naive_results = pd.DataFrame({

    "Evaluation Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Score":[
        round(nb_accuracy,4),
        round(nb_precision,4),
        round(nb_recall,4),
        round(nb_f1,4)
    ]

})

print("\nGaussian Naïve Bayes Performance")

display(naive_results)


# STEP 4 : CLASSIFICATION REPORT


print("\nClassification Report")

print(classification_report(y_test, y_pred_nb))


# GRAPH 1 : CONFUSION MATRIX


fig, ax = plt.subplots(figsize=(8,6))

ConfusionMatrixDisplay.from_predictions(

    y_test,

    y_pred_nb,

    cmap="Purples",

    colorbar=False,

    ax=ax

)

plt.title("Confusion Matrix - Gaussian Naïve Bayes")

plt.grid(False)

plt.show()


# GRAPH 2 : MODEL PERFORMANCE


metric_colors = [

    "#3498DB",
    "#E67E22",
    "#27AE60",
    "#C0392B"

]

plt.figure(figsize=(8,5))

bars = plt.bar(

    naive_results["Evaluation Metric"],

    naive_results["Score"],

    color=metric_colors

)

plt.ylim(0,1.05)

plt.title("Gaussian Naïve Bayes Performance Metrics")

plt.ylabel("Score")

plt.grid(axis="y", alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        f"{bar.get_height():.3f}",

        ha="center",

        fontsize=10

    )

plt.show()


# STEP 5 : SAMPLE PREDICTIONS


prediction_table = pd.DataFrame({

    "Actual Department": y_test.values[:15],

    "Predicted Department": y_pred_nb[:15]

})

print("\nSample Predictions")

display(prediction_table)


# STEP 6 : EXECUTION SUMMARY


execution_summary = pd.DataFrame({

    "Description":[

        "Training Time (Seconds)",

        "Prediction Time (Seconds)",

        "Training Records",

        "Testing Records"

    ],

    "Value":[

        round(training_time_nb,4),

        round(prediction_time_nb,4),

        X_train.shape[0],

        X_test.shape[0]

    ]

})

print("\nExecution Summary")

display(execution_summary)


# STEP 7 : STORE RESULTS FOR MODEL COMPARISON


model_results.append({

    "Model":"Gaussian Naïve Bayes",

    "Accuracy":nb_accuracy,

    "Precision":nb_precision,

    "Recall":nb_recall,

    "F1 Score":nb_f1,

    "Training Time":training_time_nb,

    "Prediction Time":prediction_time_nb,

    "Cross Validation":None

})

print("="*100)
print("SECTION 10 COMPLETED")
print("="*100)


In [ ]:
# SECTION 11 : MACHINE LEARNING MODEL COMPARISON


print("="*100)
print("SECTION 11 : MACHINE LEARNING MODEL COMPARISON")
print("="*100)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# STEP 1 : CREATE MODEL COMPARISON TABLE


comparison = pd.DataFrame(model_results)

comparison["Accuracy"] = comparison["Accuracy"].round(4)
comparison["Precision"] = comparison["Precision"].round(4)
comparison["Recall"] = comparison["Recall"].round(4)
comparison["F1 Score"] = comparison["F1 Score"].round(4)

comparison["Training Time"] = comparison["Training Time"].round(4)
comparison["Prediction Time"] = comparison["Prediction Time"].round(4)

comparison["Cross Validation"] = comparison["Cross Validation"].fillna(0)
comparison["Cross Validation"] = comparison["Cross Validation"].round(4)

print("\nOverall Model Comparison")

display(comparison)


# STEP 2 : MODEL RANKING


comparison["Rank"] = comparison["Accuracy"].rank(

    ascending=False,

    method="dense"

)

comparison = comparison.sort_values(

    by="Accuracy",

    ascending=False

)

print("\nModel Ranking")

display(

    comparison[

        [

            "Rank",

            "Model",

            "Accuracy",

            "Precision",

            "Recall",

            "F1 Score"

        ]

    ]

)


# STEP 3 : BEST MODEL


best_model = comparison.iloc[0]

print("\nBest Performing Model")

display(best_model)


# GRAPH 1 : ACCURACY COMPARISON


plt.figure(figsize=(10,6))

colors = [

    "#2E86AB",

    "#F6AA1C",

    "#A23B72",

    "#1B998B"

]

bars = plt.bar(

    comparison["Model"],

    comparison["Accuracy"],

    color=colors

)

plt.title(

    "Comparison of Classification Accuracy",

    fontsize=14,

    fontweight="bold"

)

plt.ylabel("Accuracy")

plt.ylim(0,1.05)

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        round(bar.get_height(),3),

        ha="center",

        fontsize=10

    )

plt.show()


# GRAPH 2 : PERFORMANCE METRIC COMPARISON


x = np.arange(len(comparison))

width = 0.2

plt.figure(figsize=(12,6))

plt.bar(

    x-width,

    comparison["Precision"],

    width,

    label="Precision",

    color="#FF6F61"

)

plt.bar(

    x,

    comparison["Recall"],

    width,

    label="Recall",

    color="#6B5B95"

)

plt.bar(

    x+width,

    comparison["F1 Score"],

    width,

    label="F1 Score",

    color="#88B04B"

)

plt.xticks(

    x,

    comparison["Model"],

    rotation=15

)

plt.ylim(0,1.05)

plt.ylabel("Score")

plt.title(

    "Comparison of Precision, Recall and F1 Score",

    fontsize=14,

    fontweight="bold"

)

plt.legend()

plt.grid(axis="y",alpha=0.3)

plt.show()


# STEP 4 : COMPUTATIONAL PERFORMANCE


execution = comparison[[

    "Model",

    "Training Time",

    "Prediction Time",

    "Cross Validation"

]]

print("\nComputational Performance")

display(execution)


# STEP 5 : FINAL PERFORMANCE SUMMARY


summary = pd.DataFrame({

    "Description":[

        "Best Model",

        "Highest Accuracy",

        "Highest Precision",

        "Highest Recall",

        "Highest F1 Score"

    ],

    "Value":[

        comparison.iloc[0]["Model"],

        comparison["Accuracy"].max(),

        comparison["Precision"].max(),

        comparison["Recall"].max(),

        comparison["F1 Score"].max()

    ]

})

print("\nPerformance Summary")

display(summary)

print("="*100)
print("SECTION 11 COMPLETED")
print("="*100)

In [ ]:
# SECTION 12 : GLOBAL EXPLAINABLE AI USING RANDOM FOREST FEATURE IMPORTANCE


print("="*100)
print("SECTION 12 : GLOBAL EXPLAINABLE AI USING RANDOM FOREST FEATURE IMPORTANCE")
print("="*100)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# STEP 1 : EXTRACT FEATURE IMPORTANCE


print("\nExtracting Feature Importance...")

importance = pd.DataFrame({

    "Feature":X.columns,

    "Importance":rf_model.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

).reset_index(drop=True)

importance["Rank"] = importance.index + 1

top10 = importance.head(10)

print("\nTop 10 Important Features")

display(top10)


# GRAPH 1 : FEATURE IMPORTANCE


plt.figure(figsize=(11,6))

colors = [

    "#1F77B4",

    "#FF7F0E",

    "#2CA02C",

    "#D62728",

    "#9467BD",

    "#8C564B",

    "#E377C2",

    "#7F7F7F",

    "#BCBD22",

    "#17BECF"

]

bars = plt.barh(

    top10["Feature"],

    top10["Importance"],

    color=colors

)

plt.gca().invert_yaxis()

plt.title(

    "Top 10 Random Forest Feature Importance",

    fontsize=15,

    fontweight="bold"

)

plt.xlabel("Importance Score")

plt.grid(axis="x",alpha=0.3)

for i,v in enumerate(top10["Importance"]):

    plt.text(

        v,

        i,

        f"{v:.4f}",

        va="center",

        fontsize=9

    )

plt.show()


# GRAPH 2 : FEATURE IMPORTANCE DISTRIBUTION


plt.figure(figsize=(8,5))

plt.hist(

    importance["Importance"],

    bins=10,

    color="#9B59B6",

    edgecolor="black"

)

plt.title(

    "Distribution of Feature Importance Scores",

    fontsize=14,

    fontweight="bold"

)

plt.xlabel("Importance")

plt.ylabel("Number of Features")

plt.grid(alpha=0.3)

plt.show()


# STEP 2 : CUMULATIVE IMPORTANCE


importance["Cumulative Importance"] = importance["Importance"].cumsum()

display(importance.head(15))


# GRAPH 3 : CUMULATIVE FEATURE IMPORTANCE


plt.figure(figsize=(10,5))

plt.plot(

    importance["Rank"],

    importance["Cumulative Importance"],

    marker="o",

    linewidth=2

)

plt.title(

    "Cumulative Feature Importance",

    fontsize=14,

    fontweight="bold"

)

plt.xlabel("Feature Rank")

plt.ylabel("Cumulative Importance")

plt.grid(alpha=0.3)

plt.show()


# STEP 3 : SUMMARY


summary = pd.DataFrame({

    "Description":[

        "Model",

        "Explainability Technique",

        "Number of Features",

        "Most Important Feature",

        "Highest Importance"

    ],

    "Value":[

        "Random Forest",

        "Feature Importance",

        len(X.columns),

        top10.iloc[0]["Feature"],

        round(top10.iloc[0]["Importance"],4)

    ]

})

print("\nFeature Importance Summary")

display(summary)

print("="*100)
print("SECTION 12 COMPLETED")
print("="*100)

In [ ]:
# SECTION 13 : LOCAL EXPLAINABLE AI USING LIME


print("="*100)
print("SECTION 13 : LOCAL EXPLAINABLE AI USING LIME")
print("="*100)

import lime
import lime.lime_tabular
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# STEP 1 : CREATE LIME EXPLAINER


print("\nCreating LIME Explainer...")

lime_explainer = lime.lime_tabular.LimeTabularExplainer(

    training_data=np.array(X_train),

    feature_names=list(X_train.columns),

    class_names=[str(c) for c in sorted(y.unique())],

    mode="classification",

    discretize_continuous=True,

    random_state=42

)

print("LIME Explainer Created Successfully")


# STEP 2 : SELECT SAMPLE


sample_index = 0

sample = X_test.iloc[sample_index]

actual_class = y_test.iloc[sample_index]

prediction = rf_model.predict(

    sample.values.reshape(1,-1)

)[0]

probability = rf_model.predict_proba(

    sample.values.reshape(1,-1)

)[0]

confidence = probability.max()

print("\nPrediction Generated Successfully")


# STEP 3 : GENERATE LIME EXPLANATION


print("\nGenerating Explanation...")

lime_exp = lime_explainer.explain_instance(

    sample.values,

    rf_model.predict_proba,

    num_features=10,

    top_labels=1

)

print("Explanation Completed")


# STEP 4 : EXTRACT THE CORRECT LABEL


label = lime_exp.available_labels()[0]

lime_features = lime_exp.as_map()[label]

lime_table = pd.DataFrame(

    [

        (

            X_train.columns[index],

            weight

        )

        for index,weight in lime_features

    ],

    columns=[

        "Feature",

        "Contribution"

    ]

)

lime_table = lime_table.sort_values(

    by="Contribution",

    ascending=False

)

lime_table["Contribution"] = lime_table["Contribution"].round(4)

print("\nTop LIME Features")

display(lime_table)


# GRAPH 1 : FEATURE CONTRIBUTION


colors = [

    "#2ECC71" if x>0 else "#E74C3C"

    for x in lime_table["Contribution"]

]

plt.figure(figsize=(10,6))

bars = plt.barh(

    lime_table["Feature"],

    lime_table["Contribution"],

    color=colors

)

plt.gca().invert_yaxis()

plt.title(

    "LIME Feature Contribution",

    fontsize=14,

    fontweight="bold"

)

plt.xlabel("Contribution")

plt.grid(axis="x",alpha=0.3)

for i,v in enumerate(lime_table["Contribution"]):

    plt.text(

        v,

        i,

        f"{v:.3f}",

        va="center"

    )

plt.show()


# STEP 5 : PREDICTION PROBABILITY


probability_df = pd.DataFrame({

    "Class":rf_model.classes_.astype(str),

    "Probability":probability

})

probability_df = probability_df.sort_values(

    by="Probability",

    ascending=False

)

print("\nPrediction Probability")

display(probability_df)


# GRAPH 2 : CLASS PROBABILITY


plt.figure(figsize=(10,5))

bars = plt.bar(

    probability_df["Class"],

    probability_df["Probability"],

    color="#3498DB"

)

plt.title(

    "Prediction Probability Distribution",

    fontsize=14,

    fontweight="bold"

)

plt.xlabel("Department")

plt.ylabel("Probability")

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        f"{bar.get_height():.2f}",

        ha="center",

        fontsize=8

    )

plt.show()


# STEP 6 : INTERPRETATION SUMMARY


summary = pd.DataFrame({

    "Description":[

        "Explainability Method",

        "Machine Learning Model",

        "Actual Class",

        "Predicted Class",

        "Prediction Confidence",

        "Most Influential Feature"

    ],

    "Value":[

        "LIME",

        "Random Forest",

        actual_class,

        prediction,

        round(confidence,4),

        lime_table.iloc[0]["Feature"]

    ]

})

print("\nInterpretation Summary")

display(summary)

print("="*100)
print("SECTION 13 COMPLETED")
print("="*100)

In [ ]:
# SECTION 14 : IMPLEMENTATION SUMMARY AND BUSINESS DECISION SUPPORT


print("="*100)
print("SECTION 14 : IMPLEMENTATION SUMMARY AND BUSINESS DECISION SUPPORT")
print("="*100)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


# STEP 1 : MODEL PERFORMANCE SUMMARY


print("\nPreparing Model Performance Summary...")

comparison = pd.DataFrame(model_results)

comparison = comparison.sort_values(

    by="Accuracy",

    ascending=False

).reset_index(drop=True)

comparison["Rank"] = comparison.index + 1

display(comparison)


# STEP 2 : BEST MODEL


best_model = comparison.iloc[0]

print("\nBest Performing Model")

display(best_model)


# GRAPH 1 : MODEL ACCURACY COMPARISON


plt.figure(figsize=(10,6))

colors = [

    "#1F77B4",

    "#FF7F0E",

    "#2CA02C",

    "#D62728"

]

bars = plt.bar(

    comparison["Model"],

    comparison["Accuracy"],

    color=colors[:len(comparison)]

)

plt.title(

    "Comparison of Machine Learning Models",

    fontsize=14,

    fontweight="bold"

)

plt.ylabel("Accuracy")

plt.ylim(0,1.05)

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        f"{bar.get_height():.3f}",

        ha="center",

        fontsize=10

    )

plt.show()


# STEP 3 : TOP BUSINESS FEATURES


print("\nMost Influential Business Features")

display(top10)


# GRAPH 2 : BUSINESS FEATURE IMPORTANCE


plt.figure(figsize=(10,6))

bars = plt.barh(

    top10["Feature"],

    top10.iloc[:,1],

    color=[

        "#16A085",

        "#27AE60",

        "#2980B9",

        "#8E44AD",

        "#E67E22",

        "#D35400",

        "#2ECC71",

        "#F39C12",

        "#3498DB",

        "#C0392B"

    ]

)

plt.gca().invert_yaxis()

plt.title(

    "Top Business Features Influencing Classification",

    fontsize=14,

    fontweight="bold"

)

plt.xlabel("Importance")

plt.grid(axis="x",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_width(),

        bar.get_y()+bar.get_height()/2,

        f"{bar.get_width():.3f}",

        va="center",

        fontsize=9

    )

plt.show()


# STEP 4 : BUSINESS INSIGHTS


business_insights = pd.DataFrame({

    "Business Insight":[

        "Best Performing Machine Learning Model",

        "Highest Classification Accuracy",

        "Most Influential Feature",

        "Explainability Method",

        "Primary Retail Benefit"

    ],

    "Finding":[

        best_model["Model"],

        round(best_model["Accuracy"],4),

        top10.iloc[0]["Feature"],

        "LIME + Random Forest Feature Importance",

        "Improved Retail Financial Decision Support"

    ]

})

print("\nBusiness Insights")

display(business_insights)


# STEP 5 : IMPLEMENTATION SUMMARY


summary = pd.DataFrame({

    "Implementation Stage":[

        "Data Collection",

        "Data Cleaning",

        "Feature Engineering",

        "Natural Language Processing",

        "Machine Learning",

        "Explainable AI",

        "Business Decision Support"

    ],

    "Status":[

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed",

        "Completed"

    ]

})

print("\nImplementation Progress Summary")

display(summary)

print("="*100)
print("SECTION 14 COMPLETED")
print("="*100)

In [ ]:
# SECTION 15 : MODEL EXPORT, REPRODUCIBILITY AND FINAL VALIDATION


print("="*100)
print("SECTION 15 : MODEL EXPORT, REPRODUCIBILITY AND FINAL VALIDATION")
print("="*100)

import joblib
import os
import pandas as pd
import numpy as np


# STEP 1 : CREATE EXPORT FOLDER


folder_name = "Saved_Model"

if not os.path.exists(folder_name):

    os.makedirs(folder_name)

print("Folder Created Successfully")


# STEP 2 : SAVE RANDOM FOREST MODEL


joblib.dump(

    rf_model,

    folder_name + "/random_forest_model.pkl"

)

print("Random Forest Model Saved")


# STEP 3 : SAVE SCALER


joblib.dump(

    scaler,

    folder_name + "/feature_scaler.pkl"

)

print("Feature Scaler Saved")


# STEP 4 : SAVE FEATURE LIST


feature_list = list(X.columns)

joblib.dump(

    feature_list,

    folder_name + "/feature_names.pkl"

)

print("Feature Names Saved")


# STEP 5 : LOAD MODEL AGAIN


loaded_model = joblib.load(

    folder_name + "/random_forest_model.pkl"

)

print("Saved Model Loaded Successfully")


# STEP 6 : VERIFY MODEL


loaded_prediction = loaded_model.predict(

    X_test.iloc[:100]

)

verification_accuracy = np.mean(

    loaded_prediction == y_test.iloc[:100]

)

verification = pd.DataFrame({

    "Description":[

        "Saved Model",

        "Verification Samples",

        "Reload Accuracy"

    ],

    "Value":[

        "Random Forest",

        100,

        round(verification_accuracy,4)

    ]

})

print("\nModel Verification")

display(verification)


# STEP 7 : SAVED FILES


saved_files = pd.DataFrame({

    "File Name":[

        "random_forest_model.pkl",

        "feature_scaler.pkl",

        "feature_names.pkl"

    ],

    "Purpose":[

        "Machine Learning Model",

        "Feature Scaling",

        "Input Features"

    ]

})

print("\nSaved Files")

display(saved_files)


# GRAPH 1 : MODEL VERIFICATION


plt.figure(figsize=(6,5))

bars = plt.bar(

    ["Reload Accuracy"],

    [verification_accuracy],

    color="#2ECC71"

)

plt.title(

    "Reloaded Model Verification",

    fontsize=14,

    fontweight="bold"

)

plt.ylim(0,1.05)

plt.ylabel("Accuracy")

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        f"{bar.get_height():.3f}",

        ha="center"

    )

plt.show()


# GRAPH 2 : PROJECT COMPLETION STATUS


stages = [

    "EDA",

    "Cleaning",

    "Features",

    "NLP",

    "ML",

    "XAI",

    "Deployment"

]

status = [100,100,100,100,100,100,100]

plt.figure(figsize=(10,5))

bars = plt.bar(

    stages,

    status,

    color=[

        "#3498DB",

        "#2ECC71",

        "#9B59B6",

        "#F39C12",

        "#E74C3C",

        "#16A085",

        "#34495E"

    ]

)

plt.title(

    "Implementation Completion Status",

    fontsize=14,

    fontweight="bold"

)

plt.ylabel("Completion (%)")

plt.ylim(0,110)

plt.grid(axis="y",alpha=0.3)

for bar in bars:

    plt.text(

        bar.get_x()+bar.get_width()/2,

        bar.get_height(),

        "100%",

        ha="center"

    )

plt.show()


# STEP 8 : FINAL IMPLEMENTATION SUMMARY


summary = pd.DataFrame({

    "Description":[

        "Dataset",

        "Best Model",

        "Explainability",

        "Model Export",

        "Implementation Status"

    ],

    "Value":[

        "Walmart M5 Retail Dataset",

        "Random Forest",

        "Feature Importance + LIME",

        "Completed",

        "Completed"

    ]

})

print("\nFinal Implementation Summary")

display(summary)

print("="*100)
print("SECTION 15 COMPLETED")
print("="*100)